In [6]:
import pandas as pd
from datetime import timedelta
import glob
import os

In [8]:
# Define path to raw data
INPUT_DIR = 'C:/Users/ayan.pathak/Desktop/banking_assistant/data/raw'
OUTPUT_DIR = "C:/Users/ayan.pathak/Desktop/banking_assistant/data/processed"
# List of files we are interested in
files = ['account.csv', 'loan.csv', 'card.csv', 'disp.csv', 'order.csv', 'trans.csv', 'client.csv']

In [5]:
for filename in files:
    file_path = os.path.join(INPUT_DIR, filename)
    if os.path.exists(file_path):
        print(f"\n{'='*20} INSPECTING: {filename} {'='*20}")
        
        # Using separator ';' as we confirmed earlier
        df = pd.read_csv(file_path, sep=';', low_memory=False)
        
        # 1. Basic Info 
        print("--- INFO ---")
        df.info()
        
        # 2. Stats 
        print("\n--- DESCRIBE ---")
        print(df.describe().T) # Transposed for readability
        
        # 3. Categorical Check 
        # Only check columns that look like categories (object or few unique ints)
        for col in df.select_dtypes(include='object').columns:
            unique_count = df[col].nunique()
            if unique_count < 20: # Only print if it's a manageable category list
                print(f"\n--- Value Counts: {col} ---")
                print(df[col].value_counts(dropna=False))
    else:
        print(f"Warning: {filename} not found.")


==================== INSPECTING: account.csv ====================
--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   account_id   4500 non-null   int64 
 1   district_id  4500 non-null   int64 
 2   frequency    4500 non-null   object
 3   date         4500 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 140.8+ KB

--- DESCRIBE ---
              count           mean           std       min        25%  \
account_id   4500.0    2786.067556   2313.811984       1.0    1182.75   
district_id  4500.0      37.310444     25.177217       1.0      13.00   
date         4500.0  951654.608667  14842.188377  930101.0  931227.00   

                  50%        75%       max  
account_id     2368.0    3552.25   11382.0  
district_id      38.0      60.00      77.0  
date         960102.0  961101.00  971229.0  

--- Value Counts: freque

In [7]:
TARGET_YEAR = 2026

# 1. GLOBAL TIME SHIFT CALCULATION
print("1. Calculating Time Shift.")
# We use trans.csv because it has the most recent dates
df_trans_ref = pd.read_csv(f'{INPUT_DIR}/trans.csv', sep=';', usecols=['date'])
max_date_int = df_trans_ref['date'].max() # e.g., 981231
max_date_obj = pd.to_datetime(str(max_date_int), format="%y%m%d", errors='coerce') # 1998-12-31

target_date = pd.Timestamp(f"{TARGET_YEAR}-12-31")
SHIFT_DELTA = target_date - max_date_obj

print(f"   Anchor Date: {max_date_obj.date()} -> Target: {target_date.date()}")
print(f"   Shift Delta: {SHIFT_DELTA.days} days")
del df_trans_ref # Free memory

# Helper for date conversion
def process_date(x):
    if pd.isna(x): return None
    # 930101 -> 1993-01-01 + Shift
    d = pd.to_datetime(str(int(x)), format="%y%m%d", errors='coerce')
    return d + SHIFT_DELTA

1. Calculating Time Shift.
   Anchor Date: 1998-12-31 -> Target: 2026-12-31
   Shift Delta: 10227 days


In [ ]:
# 2. PROCESS ACCOUNTS
print("2. Cleaning Accounts.")
df_account = pd.read_csv(f'{INPUT_DIR}/account.csv', sep=';')
df_account['date_created'] = df_account['date'].apply(process_date)

freq_map = {
    "POPLATEK MESICNE": "MONTHLY_STATEMENTS",
    "POPLATEK TYDNE": "WEEKLY_STATEMENTS",
    "POPLATEK PO OBRATU": "TRANSACTION_STATEMENTS"
}
df_account['frequency'] = df_account['frequency'].map(freq_map).fillna("UNKNOWN")
df_account.drop(columns=['date'], inplace=True)
df_account.to_csv(f'{OUTPUT_DIR}/clean_account.csv', index=False)

2. Cleaning Accounts.


In [ ]:
# 3. PROCESS LOANS
print("3. Cleaning Loans.")
df_loan = pd.read_csv(f'{INPUT_DIR}/loan.csv', sep=';')
df_loan['date_granted'] = df_loan['date'].apply(process_date)

# Mapping A/B/C/D to clear business terms
status_map = {
    'A': 'CONTRACT_FINISHED_NO_PROBLEMS',
    'B': 'CONTRACT_FINISHED_LOAN_NOT_PAID',
    'C': 'RUNNING_CONTRACT_OK',
    'D': 'RUNNING_CONTRACT_IN_DEBT'
}
df_loan['status'] = df_loan['status'].map(status_map)
df_loan.drop(columns=['date'], inplace=True)
df_loan.to_csv(f'{OUTPUT_DIR}/clean_loan.csv', index=False)

3. Cleaning Loans.


In [11]:
# 4. PROCESS CARDS
print("4. Cleaning Cards.")
df_card = pd.read_csv(f'{INPUT_DIR}/card.csv', sep=';')

# Fix format "931107 00:00:00"
def clean_card_date(x):
    if pd.isna(x): return None
    d = pd.to_datetime(str(x)[:6], format="%y%m%d", errors='coerce')
    return d + SHIFT_DELTA

df_card['issued_date'] = df_card['issued'].apply(clean_card_date)
df_card['type'] = df_card['type'].str.upper().str.strip() # 'classic' -> 'CLASSIC'
df_card.drop(columns=['issued'], inplace=True)
df_card.to_csv(f'{OUTPUT_DIR}/clean_card.csv', index=False)

4. Cleaning Cards.


In [12]:
# 5. PROCESS DISPOSITIONS (Links)
print("5. Cleaning Dispositions.")
df_disp = pd.read_csv(f'{INPUT_DIR}/disp.csv', sep=';')
disp_map = {"OWNER": "OWNER", "DISPONENT": "AUTHORIZED_USER"}
df_disp['type'] = df_disp['type'].map(disp_map)
df_disp.to_csv(f'{OUTPUT_DIR}/clean_disp.csv', index=False)

5. Cleaning Dispositions.


In [13]:
# 6. PROCESS CLIENTS (Gender Decode)
print("6. Cleaning Clients.")
df_client = pd.read_csv(f'{INPUT_DIR}/client.csv', sep=';')

def parse_client_details(bn):
    s = str(bn)
    # Check length to avoid index errors
    if len(s) < 6: return pd.Series([None, None])
    
    yy = int(s[:2])
    mm = int(s[2:4])
    dd = int(s[4:6])
    
    gender = 'M'
    if mm > 50:
        gender = 'F'
        mm = mm - 50
    
    # 19xx prefix
    full_year = 1900 + yy
    try:
        dob = pd.Timestamp(year=full_year, month=mm, day=dd)
        # Apply Time Shift to Birthdate too? 
        # Ideally YES, otherwise 50 year olds become 75 year olds.
        dob = dob + SHIFT_DELTA
    except:
        dob = None
        
    return pd.Series([dob, gender])

df_client[['birth_date', 'gender']] = df_client['birth_number'].apply(parse_client_details)
df_client.drop(columns=['birth_number'], inplace=True)
df_client.to_csv(f'{OUTPUT_DIR}/clean_client.csv', index=False)

6. Cleaning Clients.


In [14]:
# 7. PROCESS TRANSACTIONS (The Heavy Lift)
print("7. Cleaning Transactions (this may take a moment).")
df_trans = pd.read_csv(f'{INPUT_DIR}/trans.csv', sep=';', low_memory=False)

# A. Dates
df_trans['date'] = df_trans['date'].apply(process_date)

# B. Translation
trans_type_map = {'PRIJEM': 'CREDIT', 'VYDAJ': 'WITHDRAWAL', 'VYBER': 'CASH_WITHDRAWAL'}
op_map = {
    'VKLAD': 'CASH_DEPOSIT', 
    'PREVOD Z UCTU': 'TRANSFER_FROM_ACCOUNT', 
    'VYBER': 'CASH_WITHDRAWAL', 
    'PREVOD NA UCET': 'TRANSFER_TO_ACCOUNT', 
    'VYBER KARTOU': 'CREDIT_CARD_WITHDRAWAL'
}
# K_Symbol (Transaction Characteristic)
ksym_map = {
    'UROK': 'INTEREST', 'SIPO': 'HOUSEHOLD_PAYMENT', 'SLUZBY': 'SERVICE_FEE',
    'POJISTNE': 'INSURANCE', 'DUCHOD': 'PENSION', 'UVER': 'LOAN_PAYMENT',
    'SANKC. UROK': 'SANCTION_INTEREST', ' ': 'MISC_PAYMENT'
}

df_trans['type'] = df_trans['type'].map(trans_type_map).fillna('UNKNOWN')
df_trans['operation'] = df_trans['operation'].map(op_map)
df_trans['k_symbol'] = df_trans['k_symbol'].map(ksym_map)

# C. Handling Nulls (The "Fix" from our inspection)
# If Operation is Null but Type is CREDIT -> likely Interest or Automatic Credit
df_trans['operation'] = df_trans['operation'].fillna('SYSTEM_AUTO') 
df_trans['k_symbol'] = df_trans['k_symbol'].fillna('GENERAL')
df_trans['bank'] = df_trans['bank'].fillna('')
df_trans['account'] = df_trans['account'].fillna(0).astype(int) # Partner account

# Save
df_trans.to_csv(f'{OUTPUT_DIR}/clean_trans.csv', index=False)

7. Cleaning Transactions (this may take a moment).


In [15]:
# 8. PROCESS ORDERS
print("8. Cleaning Orders.")
df_order = pd.read_csv(f'{INPUT_DIR}/order.csv', sep=';')

# Fix " " k_symbol
df_order['k_symbol'] = df_order['k_symbol'].map(ksym_map).fillna('GENERAL')
df_order.to_csv(f'{OUTPUT_DIR}/clean_order.csv', index=False)

8. Cleaning Orders.


In [16]:
found_issues = False

for filename in files:
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Check for Nulls
        nulls = df.isnull().sum()
        nulls = nulls[nulls > 0] # Filter only columns with missing data
        
        if not nulls.empty:
            found_issues = True
            print(f"\n[!] ALERT: {filename} still has nulls:")
            print(nulls)
            
            # Specific check for Client table which is most fragile
            if filename == 'clean_client.csv':
                print("(These likely represent invalid birth_numbers in the source)")
        else:
            print(f"[OK] {filename} is clean.")

if not found_issues:
    print("\nSUCCESS: All files are 100% clean and ready for Database Injection.")


SUCCESS: All files are 100% clean and ready for Database Injection.
